In [0]:
%sql
describe workspace.pj_sales.tb_fact_sale_gold

In [0]:
describe workspace.pj_sales.tb_dim_date_gold

In [0]:
select
   tfs.invoice_id
  ,tfs.part_total_price
  ,tfs.part_profit
  ,tfs.part_quantity
  ,tdd.date
  ,tdd.month
  ,tdd.quarter_of_year
  ,tdd.year
from workspace.pj_sales.tb_fact_sale_gold tfs
inner join workspace.pj_sales.tb_dim_date_gold tdd
  on tdd._sk_date = tfs._sk_date
order by tdd.date asc

In [0]:
select
  count(distinct tfs.invoice_id)                                                  as sales_quantity
  ,round(sum(tfs.part_total_price), 2)                                            as billing
  ,round(sum(tfs.part_profit), 2)                                                 as profit
  ,round(sum(tfs.part_quantity), 2)                                               as product_quantity
  ,tdd.month
  ,tdd.year
from workspace.pj_sales.tb_fact_sale_gold tfs
inner join workspace.pj_sales.tb_dim_date_gold tdd
  on tdd._sk_date = tfs._sk_date
group by 
    tdd.year
  ,tdd.month


In [0]:
with
  aggregated as (
    select
      count(distinct tfs.invoice_id)                                                  as sales_quantity
      ,round(sum(tfs.part_total_price), 2)                                            as billing
      ,round(sum(tfs.part_profit), 2)                                                 as profit
      ,round(sum(tfs.part_quantity), 2)                                               as product_quantity
      ,tdd.month
      ,tdd.year
    from workspace.pj_sales.tb_fact_sale_gold tfs
    inner join workspace.pj_sales.tb_dim_date_gold tdd
      on tdd._sk_date = tfs._sk_date
    group by 
        tdd.year
      ,tdd.month
  )
select
  *
  ,ntile(3) over (partition by year order by sales_quantity desc)                  as sales_performance_group
  ,lag(sales_quantity) over (partition by year order by month)                     as prev_month_sales
from aggregated


In [0]:
with
  aggregated as (
    select
      count(distinct tfs.invoice_id)                                                  as sales_quantity
      ,round(sum(tfs.part_total_price), 2)                                            as billing
      ,round(sum(tfs.part_profit), 2)                                                 as profit
      ,round(sum(tfs.part_quantity), 2)                                               as product_quantity
      ,tdd.month
      ,tdd.year
    from workspace.pj_sales.tb_fact_sale_gold tfs
    inner join workspace.pj_sales.tb_dim_date_gold tdd
      on tdd._sk_date = tfs._sk_date
    group by 
        tdd.year
      ,tdd.month
  ),
  terciles as (
    select
      *
      ,ntile(3) over (partition by year order by sales_quantity desc)                  as sales_performance_group
      ,lag(sales_quantity) over (partition by year order by month)                     as prev_month_sales
    from aggregated
  )
select
  t.month
  ,t.year
  ,t.sales_quantity
  ,t.billing
  ,t.profit
  ,t.product_quantity
  ,round((t.sales_quantity - t.prev_month_sales) / nullif(t.prev_month_sales, 0), 2)   as sales_growth_mom
  ,dense_rank() over (partition by t.year order by t.billing        desc)              as rank_by_billing_per_month
  ,dense_rank() over (partition by t.year order by t.profit         desc)              as rank_by_profit_per_month
  ,dense_rank() over (partition by t.year order by t.sales_quantity desc)              as rank_by_sales_quantity_per_month
  ,case
    when t.sales_performance_group = 1                                                 then 'top'
    when t.sales_performance_group = 2                                                 then 'middle'
    else                                                                                    'low'
  end                                                                                  as sales_performance_tier
from terciles t
order by
   year
  ,month

In [0]:
create or replace view workspace.pj_sales.vw_sales_performance_by_time as (
  with
    aggregated as (
      select
        count(distinct tfs.invoice_id)                                                  as sales_quantity
        ,round(sum(tfs.part_total_price), 2)                                            as billing
        ,round(sum(tfs.part_profit), 2)                                                 as profit
        ,round(sum(tfs.part_quantity), 2)                                               as product_quantity
        ,tdd.month
        ,tdd.year
      from workspace.pj_sales.tb_fact_sale_gold tfs
      inner join workspace.pj_sales.tb_dim_date_gold tdd
        on tdd._sk_date = tfs._sk_date
      group by 
          tdd.year
        ,tdd.month
    ),
    terciles as (
      select
        *
        ,ntile(3) over (partition by year order by sales_quantity desc)                  as sales_performance_group
        ,lag(sales_quantity) over (partition by year order by month)                     as prev_month_sales
      from aggregated
    )
  select
    t.month
    ,t.year
    ,t.sales_quantity
    ,t.billing
    ,t.profit
    ,t.product_quantity
    ,round((t.sales_quantity - t.prev_month_sales) / nullif(t.prev_month_sales, 0), 2)   as sales_growth_mom
    ,dense_rank() over (partition by t.year order by t.billing        desc)              as rank_by_billing_per_month
    ,dense_rank() over (partition by t.year order by t.profit         desc)              as rank_by_profit_per_month
    ,dense_rank() over (partition by t.year order by t.sales_quantity desc)              as rank_by_sales_quantity_per_month
    ,case
      when t.sales_performance_group = 1                                                 then 'top'
      when t.sales_performance_group = 2                                                 then 'middle'
      else                                                                                    'low'
    end                                                                                  as sales_performance_tier
  from terciles t
)